<a href="https://colab.research.google.com/github/tanishq25iitroorkee-netizen/HMM_project/blob/main/HMM_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                                  [0.0, 0.9, 0.1, 0.0, 0.0],
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]])
emission_nuc_codes = {'A': 0,
                      'C': 1,
                      'G': 2,
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00],
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [ ]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)

-43.89740030179307

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)

Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides.

             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .]
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .]
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

It is important to remember that you will be working in the log scale.

In [ ]:
# Initiate two matrices:
# viterbi_value_matrix: to store the values described in the documentation above
# viterbi_trace_matrix: to store the path the lead to the the maximum value in each cell
# For example, the first column of viterbi_trace_matrix will be
# [0] indicating start state released `C`: even though not possible - but we just initiate
# [1] indicating Exon state released `C`:
# [2] indicating 5'ss state released `C`: even though not possible - but we just initiate
# [3] indicating Intron state released `C`:
# [4] indicating end state released `C`: even though not possible - but we just initiate

Implementation of Viterbi Algorithm

Write a function calculate_prob_for_a_node() that populate a single cell in the matrix. The function will return two values:

the maximum value, for example, look at the 2nd row, 2nd column in the matrix: max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T). If the probability for s-E-C-E-T is highest (lets say X), then the function should return X
AND

The index of that maximum value described in the first point: so index of X is 1 (recall that Python works on the 0-based index system)
Populate viterbi_value_matrix with X for row 2 and col 2

Populate viterbi_trace_matrix with 1 for row 2 and col 2

In [ ]:
# Write for loops to iterate over the whole Viterbi Value matrix.
# Each time, call the function

In [ ]:
# Write a function to trace the state path that gave the maximum probability.
# This will be the final result.


# HINT: You should first find the maximum value in the last column of `viterbi_value_matrix`,
# because that is the one with the largest probability.
# The index of that value is the state of the last nucleotide.

In [1]:
import numpy as np
import math

def run_viterbi(obs_seq, states_dict, trans_prob, emiss_prob, emiss_codes):
    """
    Implementation of the Viterbi algorithm using log-probabilities.
    """
    n_obs = len(obs_seq)
    n_states = len(states_dict)

    # Initialize Viterbi Value and Trace matrices
    # Using -inf for log(0) probabilities
    viterbi_val = np.full((n_states, n_obs), -np.inf)
    viterbi_trace = np.zeros((n_states, n_obs), dtype=int)

    # 1. Initialization Step (t=0)
    # Start state 's' is index 0. Transition from 's' to state i.
    # In the provided notebook, start state 's' (0) transitions to 'E' (1) with prob 1.0
    first_obs_idx = emiss_codes[obs_seq[0]]
    for s_idx in range(n_states):
        t_prob = trans_prob[0, s_idx]
        e_prob = emiss_prob[s_idx, first_obs_idx]

        if t_prob > 0 and e_prob > 0:
            viterbi_val[s_idx, 0] = math.log(t_prob) + math.log(e_prob)

    # 2. Recursion Step (t=1 to n_obs-1)
    for t in range(1, n_obs):
        obs_idx = emiss_codes[obs_seq[t]]
        for curr_s in range(n_states):
            # Calculate potential paths from all previous states to curr_s
            e_prob = emiss_prob[curr_s, obs_idx]
            if e_prob == 0:
                continue

            log_e = math.log(e_prob)

            # find max_{prev_s} (Viterbi[prev_s, t-1] + log(trans[prev_s, curr_s]))
            best_prev_val = -np.inf
            best_prev_idx = 0

            for prev_s in range(n_states):
                t_prob = trans_prob[prev_s, curr_s]
                if t_prob > 0:
                    val = viterbi_val[prev_s, t-1] + math.log(t_prob)
                    if val > best_prev_val:
                        best_prev_val = val
                        best_prev_idx = prev_s

            if best_prev_val != -np.inf:
                viterbi_val[curr_s, t] = best_prev_val + log_e
                viterbi_trace[curr_s, t] = best_prev_idx

    # 3. Termination/Backtracking
    # Find best final state (excluding start state 's' usually)
    best_last_state = np.argmax(viterbi_val[:, n_obs - 1])
    max_log_prob = viterbi_val[best_last_state, n_obs - 1]

    # Backtrack
    path = [best_last_state]
    for t in range(n_obs - 1, 0, -1):
        path.append(viterbi_trace[path[-1], t])

    path.reverse()
    return path, max_log_prob, viterbi_val

In [4]:
import sys
import os
import math
import numpy as np
import pandas as pd

# 1. Model Definitions
states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                                  [0.0, 0.9, 0.1, 0.0, 0.0],
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]])
emission_nuc_codes = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00],
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# 2. Viterbi Algorithm Implementation
def run_viterbi(obs_seq, states_dict, trans_prob, emiss_prob, emiss_codes):
    n_obs = len(obs_seq)
    n_states = len(states_dict)
    viterbi_val = np.full((n_states, n_obs), -np.inf)
    viterbi_trace = np.zeros((n_states, n_obs), dtype=int)

    first_obs_idx = emiss_codes[obs_seq[0]]
    for s_idx in range(n_states):
        t_prob = trans_prob[0, s_idx]
        e_prob = emiss_prob[s_idx, first_obs_idx]
        if t_prob > 0 and e_prob > 0:
            viterbi_val[s_idx, 0] = math.log(t_prob) + math.log(e_prob)

    for t in range(1, n_obs):
        obs_idx = emiss_codes[obs_seq[t]]
        for curr_s in range(n_states):
            e_prob = emiss_prob[curr_s, obs_idx]
            if e_prob == 0: continue
            log_e = math.log(e_prob)

            best_prev_val = -np.inf
            best_prev_idx = 0
            for prev_s in range(n_states):
                t_prob = trans_prob[prev_s, curr_s]
                if t_prob > 0:
                    val = viterbi_val[prev_s, t-1] + math.log(t_prob)
                    if val > best_prev_val:
                        best_prev_val = val
                        best_prev_idx = prev_s

            if best_prev_val != -np.inf:
                viterbi_val[curr_s, t] = best_prev_val + log_e
                viterbi_trace[curr_s, t] = best_prev_idx

    best_last_state = np.argmax(viterbi_val[:, n_obs - 1])
    path = [best_last_state]
    for t in range(n_obs - 1, 0, -1):
        path.append(viterbi_trace[path[-1], t])
    path.reverse()
    return path, viterbi_val[best_last_state, n_obs - 1], viterbi_val

# 3. Execution and Results
path_indices, final_log_prob, v_matrix = run_viterbi(
    query_sequence, states, state_transition_prob, emission_probs, emission_nuc_codes
)

most_probable_path = "".join([id2state[i] for i in path_indices])
print(f"Query Sequence: {query_sequence}")
print(f"Viterbi Path:  {most_probable_path}")
print(f"Final Log Probability: {final_log_prob}")

df_viterbi = pd.DataFrame(v_matrix, index=[id2state[i] for i in range(len(states))], columns=list(query_sequence))
display(df_viterbi)

Query Sequence: CTTCATGTGAAAGCAGACGTAAGTCA
Viterbi Path:  EEEEEEEEEEEEEEEEEEEEEEEEEE
Final Log Probability: -38.677666280562796


,C,T,T,C,A,T,G,T,G,A,...,A,C,G,T,A,A,G,T,C,A
s,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
E,-1.386294,-2.877949,-4.369604,-5.861259,-7.352914,-8.844569,-10.336224,-11.827878,-13.319533,-14.811188,...,-25.252772,-26.744427,-28.236082,-29.727737,-31.219392,-32.711047,-34.202702,-35.694357,-37.186011,-38.677666
5,-inf,-inf,-inf,-inf,-11.159576,-inf,-11.198447,-inf,-14.181757,-18.617851,...,-29.059435,-inf,-29.098306,-inf,-35.026054,-36.517709,-35.064925,-inf,-inf,-42.484329
I,-inf,-inf,-inf,-inf,-inf,-12.075867,-14.483813,-12.114738,-14.522683,-15.098048,...,-25.539632,-27.947577,-30.355523,-30.014596,-31.036248,-32.057899,-34.465844,-35.487496,-37.895441,-38.917093
e,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
